# NB06 — Journal Club Presentation Skills

**Module:** 19 — Research Seminar and Paper Reading  
**Estimated time:** 2 hours  

---

## Motivation

A journal club presentation is a structured oral Pass-3. You read a paper, digest it, and explain it to a room of peers — including a supervisor who already knows the field. Done well, it demonstrates independent scientific thinking and the ability to situate a paper in a broader context. Done poorly, it reveals that you summarized the abstract without understanding the claims.

Monthly journal club practice using this notebook is part of the core programme cadence.

## 1. Journal Club Structure: 15–20 Minutes, 8–12 Slides

| Slide | Content | Time | Key mistake to avoid |
|-------|---------|------|---------------------|
| 1–2 | **Background** — what is the biological/computational problem? What did we know before this paper? | 3 min | Starting with the paper's methods before establishing why anyone should care |
| 3 | **Gap** — what is missing from prior work? What is the paper's stated contribution? | 1 min | Skipping this — the gap is the most important part |
| 4–5 | **Approach** — what did they do? (Methods at the level of: what input, what algorithm, what output) | 3 min | Going too deep into implementation details |
| 6–7 | **Key results** — the 2–3 most important figures with your interpretation | 5 min | Reading the caption aloud instead of interpreting the figure |
| 8 | **Limitations** — what the paper does not show, what could go wrong | 2 min | Omitting this — it's where you show critical thinking |
| 9 | **Open questions** — what you would do next, what is unresolved | 2 min | Saying "this is a great paper" instead of asking a real question |

## 2. Slide Design Principles

- Maximum 40 words per slide (excluding figure captions)
- Every results slide should have at least one figure
- Never put a table of numbers in a 15-minute talk — it cannot be read
- Title each slide with the *conclusion*, not the topic. "STAR aligns faster than HISAT2 on all datasets" not "Alignment benchmark"
- Figures must have axis labels visible from the back of a room

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
import matplotlib.pyplot as plt
import numpy as np


@dataclass
class Slide:
    section: str       # background, gap, approach, results, limitations, questions
    title: str
    word_count: int
    has_figure: bool
    notes: Optional[str] = None


class PresentationAudit:
    """Audit a slide outline for journal club compliance."""

    REQUIRED_SECTIONS = {'background', 'gap', 'approach', 'results', 'limitations', 'questions'}
    MAX_WORDS_PER_SLIDE = 40
    MIN_SLIDES = 8
    MAX_SLIDES = 12

    def __init__(self, slides: List[Slide], paper_title: str = ''):
        self.slides = slides
        self.paper_title = paper_title

    def audit(self) -> dict:
        issues = []
        passes = []

        # Check slide count
        n = len(self.slides)
        if self.MIN_SLIDES <= n <= self.MAX_SLIDES:
            passes.append(f'Slide count: {n} (within {self.MIN_SLIDES}–{self.MAX_SLIDES})')
        else:
            issues.append(f'Slide count: {n} (must be {self.MIN_SLIDES}–{self.MAX_SLIDES})')

        # Check all required sections present
        present_sections = {s.section for s in self.slides}
        missing = self.REQUIRED_SECTIONS - present_sections
        if not missing:
            passes.append('All 6 required sections present')
        else:
            issues.append(f'Missing sections: {", ".join(sorted(missing))}')

        # Check word count per slide
        over_limit = [s for s in self.slides if s.word_count > self.MAX_WORDS_PER_SLIDE]
        if not over_limit:
            passes.append(f'All slides within {self.MAX_WORDS_PER_SLIDE}-word limit')
        else:
            for s in over_limit:
                issues.append(f'Slide "{s.title}": {s.word_count} words (limit {self.MAX_WORDS_PER_SLIDE})')

        # Check results slides have figures
        results_slides = [s for s in self.slides if s.section == 'results']
        results_without_fig = [s for s in results_slides if not s.has_figure]
        if not results_without_fig:
            passes.append('All results slides include a figure')
        else:
            for s in results_without_fig:
                issues.append(f'Results slide "{s.title}" has no figure')

        score = len(passes) / (len(passes) + len(issues)) * 10
        return {
            'n_slides': n,
            'passes': passes,
            'issues': issues,
            'score': round(score, 1),
            'ready': len(issues) == 0,
        }


# Build a complete slide outline for a synthetic paper
# Synthetic paper: "FastSplice: a novel splice-aware RNA-seq aligner"
slides = [
    Slide('background', 'RNA-seq enables transcriptome-wide expression profiling', 30, False,
          notes='Explain why alignment matters and what the reference genome provides'),
    Slide('background', 'Existing aligners sacrifice speed or accuracy — not both', 35, True,
          notes='Show a comparison table or figure from a prior benchmark'),
    Slide('gap', 'No aligner handles long reads and splice junctions with < 1% error at > 1M reads/min', 28, False,
          notes='This is the specific gap FastSplice claims to fill'),
    Slide('approach', 'FastSplice: FM-index seed finding + SIMD-accelerated extension', 22, True,
          notes='Architecture diagram here'),
    Slide('approach', 'Two-pass strategy: discover splice junctions in Pass 1, realign in Pass 2', 18, True,
          notes='Pipeline flowchart'),
    Slide('results', 'FastSplice achieves 1.8M reads/min with 0.8% mismatch on simulated data', 20, True,
          notes='Main benchmark figure'),
    Slide('results', 'Performance holds on 3 real datasets (human, mouse, Arabidopsis)', 15, True,
          notes='Multi-species benchmark figure'),
    Slide('limitations', 'Benchmarked only on short reads (< 150 bp); long-read performance unknown', 25, False,
          notes='Authors acknowledge this; also note: single benchmark dataset per species'),
    Slide('questions', 'Can the FM-index be replaced by a more memory-efficient structure for large genomes?', 20, False,
          notes='My question: is memory the next bottleneck as genome sizes grow?'),
]

audit = PresentationAudit(slides, paper_title='FastSplice: A Novel Splice-Aware RNA-seq Aligner')
result = audit.audit()

print(f"Presentation audit for: {audit.paper_title}")
print(f"Score: {result['score']}/10 | Ready: {result['ready']}")
print(f"\nPasses ({len(result['passes'])})")
for p in result['passes']:
    print(f"  PASS: {p}")
print(f"\nIssues ({len(result['issues'])})")
for issue in result['issues']:
    print(f"  ISSUE: {issue}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Journal Club Presentation Analysis', fontsize=13, fontweight='bold')

# Panel 1: Presentation timeline (minutes per section)
ax1 = axes[0]
section_times = {
    'Background': 3, 'Gap': 1, 'Approach': 3,
    'Key Results': 5, 'Limitations': 2, 'Open Questions': 2,
    'Discussion': 3,
}
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#009688', '#795548']
wedges, texts, autotexts = ax1.pie(
    list(section_times.values()),
    labels=list(section_times.keys()),
    autopct='%1.0f min',
    colors=colors, startangle=90
)
for at in autotexts:
    at.set_fontsize(8)
ax1.set_title('Time Allocation\n(20-min presentation)')

# Panel 2: Word count per slide (with limit line)
ax2 = axes[1]
slide_names = [f'S{i+1}: {s.section[:4]}' for i, s in enumerate(slides)]
word_counts = [s.word_count for s in slides]
bar_colors = ['#E53935' if wc > 40 else '#43A047' for wc in word_counts]
ax2.bar(range(len(slides)), word_counts, color=bar_colors, alpha=0.85)
ax2.axhline(40, color='red', linestyle='--', linewidth=1.5, label='40-word limit')
ax2.set_xticks(range(len(slides)))
ax2.set_xticklabels(slide_names, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Word count')
ax2.set_title('Words Per Slide')
ax2.legend()
ax2.grid(True, axis='y', alpha=0.3)

# Panel 3: Common mistakes heatmap
ax3 = axes[2]
mistakes = [
    'No background/motivation',
    'Skip the gap',
    'Methods too detailed',
    'Read captions aloud',
    'Missing limitations',
    'No real question',
    'Over 40 words/slide',
    'Results slide, no figure',
]
experience_levels = ['Novice', 'Intermediate', 'Expert']
freq_data = np.array([
    [0.9, 0.7, 0.2],  # No background
    [0.8, 0.5, 0.1],  # Skip gap
    [0.9, 0.6, 0.2],  # Methods too detailed
    [0.7, 0.4, 0.1],  # Read captions
    [0.8, 0.3, 0.1],  # Missing limitations
    [0.9, 0.5, 0.2],  # No real question
    [0.6, 0.4, 0.1],  # Over 40 words
    [0.5, 0.2, 0.05], # No figure
])
im = ax3.imshow(freq_data, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax3.set_xticks(range(len(experience_levels)))
ax3.set_xticklabels(experience_levels)
ax3.set_yticks(range(len(mistakes)))
ax3.set_yticklabels(mistakes, fontsize=8)
ax3.set_title('Common Mistakes by Experience\n(frequency of occurrence)')
plt.colorbar(im, ax=ax3, label='Frequency (0=never, 1=always)')

plt.tight_layout()
plt.savefig('journal_club_audit.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercises

**Exercise 1:** Choose any paper from the programme's `papers.md`. Build a slide outline using the `Slide` dataclass. Run `PresentationAudit`. What issues are flagged?

**Exercise 2:** Write the "Gap" slide (one slide, under 40 words) for Needleman-Wunsch (1970). What was missing before this paper?

**Exercise 3:** Give a 2-minute verbal explanation of a paper to a real person (or record yourself). Play back: did you cover background, gap, approach, result, and one limitation?

**Exercise 4 (reflection):** Why does titling a slide with the conclusion ("STAR is 50× faster on all datasets") communicate better than titling it with the topic ("Speed comparison")?

## Quiz

1. What are the 6 required sections of a journal club presentation?
2. What is the maximum word count per slide?
3. Why must every results slide include a figure?
4. What is the "gap" slide, and why is it the most important?
5. What makes the limitations section a demonstration of critical thinking?

## References

- Bourne, P.E. (2007). Ten simple rules for making good oral presentations. PLOS Comp Bio 3(4):e77.
- Seibert Hanson & Brown (2019). Improving journal clubs by addressing student fears. CBE Life Sci Educ 18(3):ar43.